# 00 - Prepare the cleaned RSNA dataset (Phases 1-2)

Run this **once**. It converts DICOM -> PNG (uniform CLAHE, no label leak) and builds the patient-wise, stratified YOLO dataset used by all six model notebooks.

### Before you run
1. *Add Input* -> *Competitions* -> **RSNA Pneumonia Detection Challenge** (accept the rules first). It mounts at `/kaggle/input/rsna-pneumonia-detection-challenge`.
2. GPU is optional here (CPU conversion).

In [ ]:
import os, sys
REPO_URL = "https://github.com/202201638/Graduation_project_Fully_team_2026.git"
WORK = "/kaggle/working"
os.environ["PNG_DIR"]          = f"{WORK}/png_images"
os.environ["YOLO_DATASET_DIR"] = f"{WORK}/yolo_dataset"
os.environ["ARTIFACT_DIR"]     = f"{WORK}/artifacts"
# RSNA competition data: Add Input -> Competitions -> "RSNA Pneumonia Detection Challenge"
os.environ["RSNA_DATA_DIR"]    = "/kaggle/input/rsna-pneumonia-detection-challenge"

REPO_DIR = f"{WORK}/repo"
if not os.path.exists(REPO_DIR):
    os.system(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
AI_DIR = f"{REPO_DIR}/ai"
sys.path.insert(0, AI_DIR)
os.chdir(AI_DIR)
os.system("pip install -q ultralytics torchmetrics pydicom")


In [ ]:
# None = full 26,684 images (recommended, most defensible).
# Set to 18000 for a faster, class-stratified subset (still > 15k).
MAX_IMAGES = None

from src.dataset import explore_dataset
from src.preprocessing import convert_dicom_to_png
from src.yolo_dataset import build_yolo_dataset

df = explore_dataset()
convert_dicom_to_png(df, max_images=MAX_IMAGES)   # uniform CLAHE, stratified if capped
build_yolo_dataset(df)                            # leak-free, patient-wise, stratified split


## Split summary (counts + class balance)

In [ ]:
import json
with open("/kaggle/working/artifacts/phase2_yolo_dataset_summary.json") as f:
    print(json.dumps(json.load(f), indent=2))


## Publish as a Kaggle Dataset
The cleaned dataset is at `/kaggle/working/yolo_dataset`.
1. Click **Save Version** (commit).
2. From the notebook **Output**, choose **Create Dataset** (New Dataset from notebook output) and include the `yolo_dataset/` folder. Name it e.g. `rsna-prepped-yolo-dataset`.
3. In each model notebook set `PREPPED_INPUT = "/kaggle/input/<your-dataset-slug>"`.